In [55]:
import os
import cv2
import yaml
import json
import csv
import numpy as np
from ultralytics import YOLO
from huggingface_hub import hf_hub_download
from IPython.display import display, clear_output
from PIL import Image

**LOADING THE YAML FILE**

In [ ]:
with open("/content/config.yaml", "r") as file:
    config = yaml.safe_load(file)

In [57]:
dataset_path=config["project"]["base_raw_path"]

face_config = config["face_detection"]


CONF_THRESHOLD = face_config["conf_threshold"]
IOU_THRESHOLD = face_config["iou_threshold"]
MAX_DET = face_config["max_det"]
MIN_BOX_AREA=face_config["min_box_area"]
DEVICE = face_config["device"]
model=None

**HELPER FUNCTIONS**

In [58]:
def is_valid_frame(frame):
    return frame is not None and frame.size > 0

In [59]:
def _load_model():
    global model
    if face_config["source"] == "huggingface":
        model_path = hf_hub_download(
            repo_id=face_config["repo_id"],
            filename=face_config["filename"]
        )
    else:
        model_path = face_config["model_path"]
    model = YOLO(model_path)
    print("✅ Model loaded:", model.names)

In [60]:
def detect_faces(frame_id, image):

    results = model(
        image,
        conf=CONF_THRESHOLD,
        iou=IOU_THRESHOLD,
        max_det=MAX_DET,
        device=DEVICE,
        verbose=False
    )

    detections = []

    for r in results:
        if r.boxes is None:
            continue

        boxes = r.boxes.xyxy.cpu().numpy()
        scores = r.boxes.conf.cpu().numpy()

        for box, score in zip(boxes, scores):
            x_min, y_min, x_max, y_max = box

            detections.append({
                "bbox": [
                    int(x_min),
                    int(y_min),
                    int(x_max),
                    int(y_max)
                ],
                "confidence": float(score)
            })

    return {
        "frame_id": frame_id,
        "detections": detections
    }

In [61]:
def prepare_detections_for_bytetracker(frame_data):
    detections=[]
    for det in frame_data["detections"]:
        score=det["confidence"]
        x1,y1,x2,y2=det["bbox"]

        if score<CONF_THRESHOLD:
            continue

        area=(x2-x1)*(y2-y1)
        if area<MIN_BOX_AREA:
            continue

        detections.append([x1,y1,x2,y2,score])

    if len(detections)>0:
        return np.array(detections, dtype=float)
    else:
        return np.empty((0,5), dtype=float)

**MAIN PIPELINE FUNCTION**

In [62]:
def run_detection_pipeline(dataset_path=None):
    if dataset_path is None:
        dataset_path = config["project"]["base_raw_path"]
    
    assert os.path.exists(dataset_path), f"Path not found: {dataset_path}"
    ...

    _load_model()

    all_detections=[]
    frame_files = sorted(os.listdir(dataset_path))
    for idx, fname in enumerate(frame_files):
        frame = cv2.imread(os.path.join(dataset_path, fname))
        if not is_valid_frame(frame):
            continue
        all_detections.append(detect_faces(idx, frame))
    print(f"✅ Detection done: {len(all_detections)} frames")

    prepared_detections = []
    for frame_data in all_detections:
        prepared_detections.append({
            "frame_id":   frame_data["frame_id"],
            "detections": prepare_detections_for_bytetracker(frame_data)
        })
    print(f"✅ Preparation done: {len(prepared_detections)} frames ready for tracker")

    return prepared_detections

In [63]:
if __name__ == "__main__":
    prepared_detections = run_detection_pipeline()

✅ Model loaded: {0: 'FACE'}
✅ Detection done: 2292 frames
✅ Preparation done: 2292 frames ready for tracker
